# 🤖 06 — ML Model Training & Comparison

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook trains and compares ML models for **grading error prediction**:

| Model | Algorithm | Key Strength |
|-------|-----------|-------------|
| Random Forest | Ensemble of decision trees | Handles non-linearity, robust |
| SVM (RBF) | Support Vector Machine | Good with small datasets |
| Logistic Regression | Linear classifier | Interpretable, fast |
| Gradient Boosting | Sequential boosting | High accuracy |

**Task:** Binary classification — predict whether a grading error has occurred  
**Threshold:** |AI mark − Expert mark| ≥ 2.0 marks

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
ERROR_THRESHOLD = 2.0

## 1. Load Training Data

In [ ]:
synthetic_path = Path('../data/samples/synthetic_dataset.jsonl')
records = []

with open(synthetic_path, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

print(f'Total records: {len(records)}')

# Extract feature matrix and labels
feature_keys = sorted(records[0]['features'].keys())
X = np.array([[r['features'][k] for k in feature_keys] for r in records])
y = np.array([1 if r['is_grading_error'] else 0 for r in records])

print(f'Feature matrix shape: {X.shape}')
print(f'Feature names: {feature_keys}')
print(f'\nClass distribution:')
print(f'  Non-error (0): {np.sum(y == 0)} ({np.mean(y == 0):.1%})')
print(f'  Error (1):     {np.sum(y == 1)} ({np.mean(y == 1):.1%})')
print(f'\nError threshold: |AI - Expert| >= {ERROR_THRESHOLD} marks')

## 2. Model Definitions

In [ ]:
# Define model pipelines (StandardScaler → Classifier)
models = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=100, max_depth=10, class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
            probability=True, random_state=RANDOM_STATE
        ))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C=1.0, class_weight='balanced', max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            random_state=RANDOM_STATE
        ))
    ]),
}

print(f'Models to compare: {len(models)}')
for name in models:
    print(f'  • {name}')

## 3. Cross-Validation Comparison

In [ ]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = {}

print(f'{"Model":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>10} {"ROC-AUC":>10}')
print('─' * 80)

for name, pipeline in models.items():
    results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
    cv_results[name] = results
    
    acc = results['test_accuracy'].mean()
    prec = results['test_precision'].mean()
    rec = results['test_recall'].mean()
    f1 = results['test_f1'].mean()
    auc_val = results['test_roc_auc'].mean()
    
    print(f'{name:<25} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc_val:>10.4f}')

In [ ]:
# Visualize cross-validation results
if cv_results:
    metrics_to_plot = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
    
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    fig.suptitle('5-Fold Cross-Validation Results', fontsize=16, fontweight='bold')
    
    model_names = list(cv_results.keys())
    colors = ['#4A90D9', '#E8744F', '#50C878', '#9B59B6']
    
    for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
        data = [cv_results[m][metric] for m in model_names]
        bp = ax.boxplot(data, labels=[n.split()[0] for n in model_names],
                        patch_artist=True, showmeans=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title(label, fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig('../data/results/cv_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Train Final Model & Evaluate

In [ ]:
from sklearn.model_selection import train_test_split

# Select best model based on F1 (grading error detection = recall matters)
best_name = max(cv_results, key=lambda m: cv_results[m]['test_f1'].mean())
print(f'Best model by F1: {best_name}')
print(f'  Mean F1: {cv_results[best_name]["test_f1"].mean():.4f}')
print(f'  Mean Recall: {cv_results[best_name]["test_recall"].mean():.4f}')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'\nTrain set: {X_train.shape[0]} ({np.mean(y_train):.1%} errors)')
print(f'Test set:  {X_test.shape[0]} ({np.mean(y_test):.1%} errors)')

# Train final model
final_model = models[best_name]
final_model.fit(X_train, y_train)

# Predictions
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print(f'\n{"═" * 50}')
print(f'Classification Report — {best_name}')
print(f'{"═" * 50}')
print(classification_report(y_test, y_pred, target_names=['Non-Error', 'Grading Error']))

In [ ]:
# Confusion Matrix + ROC Curve + Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Non-Error', 'Error'])
ax.set_yticklabels(['Non-Error', 'Error'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=18)

# 2. ROC Curve
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'{best_name} (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# 3. Precision-Recall Curve
ax = axes[2]
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax.plot(recall_vals, precision_vals, 'r-', linewidth=2, label=f'AP = {ap:.3f}')
ax.fill_between(recall_vals, precision_vals, alpha=0.1, color='red')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()

plt.suptitle(f'Model Evaluation — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/results/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
# Train a Random Forest specifically for feature importance
rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])
rf.fit(X_train, y_train)

importances = rf.named_steps['clf'].feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_keys)))

ax.barh(range(len(feature_keys)),
        importances[sorted_idx[::-1]],
        color=colors, edgecolor='white')
ax.set_yticks(range(len(feature_keys)))
ax.set_yticklabels([feature_keys[i] for i in sorted_idx[::-1]], fontsize=9)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')

for i, v in enumerate(importances[sorted_idx[::-1]]):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/results/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save the Best Model

In [ ]:
import joblib

# Train on full dataset for deployment
final_model.fit(X, y)

model_dir = Path('../trained_models')
model_dir.mkdir(exist_ok=True)
model_path = model_dir / 'risk_model.joblib'

# Save using the project's model class
from models import GradingRiskModel

risk_model = GradingRiskModel(
    model=final_model,
    model_name=best_name,
    feature_names=feature_keys,
)
risk_model.save(model_path)

file_size = model_path.stat().st_size / 1024
print(f'✓ Model saved: {model_path} ({file_size:.0f} KB)')
print(f'  Model: {best_name}')
print(f'  Features: {len(feature_keys)}')
print(f'  Trained on: {len(records)} records')

## Summary

| Model | F1 | Recall | Precision | ROC-AUC |
|-------|-----|--------|-----------|----------|
| Random Forest | See output | See output | See output | See output |
| SVM (RBF) | See output | See output | See output | See output |
| Logistic Regression | See output | See output | See output | See output |
| Gradient Boosting | See output | See output | See output | See output |

**Key Design Decisions:**
- `class_weight='balanced'` to handle class imbalance (~22% errors)
- Recall is prioritized — it's better to flag too many papers for review than miss real errors
- StandardScaler normalizes features before SVM/LogReg

**Next:** Proceed to `07_full_pipeline_demo.ipynb` for the end-to-end demonstration.